# 🧠 NetraEdge Training — Indian Demographics (ALL 36 Regions)
Runtime → Run All → Go to sleep → Come back in 12 hours → Models ready
---

In [ ]:
# CELL 1: Setup + GPU + Install
import subprocess, sys, os, gc, time, warnings
warnings.filterwarnings('ignore')

# GPU check
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
assert r.returncode==0, "❌ No GPU! Runtime → Change runtime type → GPU"
print(f"✅ GPU: {r.stdout.strip()}")

import torch
print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "CUDA not available!"
print(f"GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Install deps (force reinstall to avoid conflicts)
!pip install -q --force-reinstall scikit-learn numpy scipy joblib
!pip install -q torch torchvision onnx albumentations tqdm
!pip install -q retinaface-py opencv-python-headless
print("✅ All deps installed")

# Imports
import numpy as np
from pathlib import Path
from collections import Counter
from tqdm.notebook import tqdm

WORK_DIR = "/content/netraedge_training"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
os.makedirs("checkpoints", exist_ok=True)

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print("✅ Setup complete")

In [ ]:
# CELL 2: Download IndicFairFace (ALL 36 Indian regions)
if not os.path.exists("IndicFairFace"):
    os.system("git clone --depth 1 https://github.com/aarishshahmohsin/IndicFairFace.git 2>/dev/null")

# Check actual structure
indic_candidates = ["IndicFairFace/Dataset", "IndicFairFace/balanced_dataset", "IndicFairFace"]
indic_dir = None
for c in indic_candidates:
    if os.path.isdir(c):
        # Check if it has subdirectories with images
        subdirs = [d for d in os.listdir(c) if os.path.isdir(os.path.join(c, d))]
        if len(subdirs) > 5:
            indic_dir = c
            print(f"✅ IndicFairFace found at {c} ({len(subdirs)} regions)")
            break

if not indic_dir:
    print("⚠️ IndicFairFace not found or wrong structure")
    print(f"   Files in IndicFairFace/: {os.listdir('IndicFairFace') if os.path.exists('IndicFairFace') else 'N/A'}")

In [ ]:
# CELL 3: Download IMFDB via Kaggle (working method)
if not os.path.exists("IMFDB"):
    print("📥 Downloading IMFDB via Kaggle...")
    os.system("pip install -q kaggle")
    # Try Kaggle API
    result = os.system("kaggle datasets download -d ashishpatel26/indian-movie-face-database-imfdb -p /tmp/imfdb --unzip -q 2>/dev/null")
    if result == 0 and os.path.exists("/tmp/imfdb"):
        # Move to IMFDB directory
        import shutil
        # Find the actual data folder
        for item in Path("/tmp/imfdb").rglob("*"):
            if item.is_dir() and any(item.glob("*.jpg")):
                shutil.copytree(str(item), "IMFDB", dirs_exist_ok=True)
                break
        print("✅ IMFDB downloaded via Kaggle")
    else:
        # Fallback: direct download
        print("  Trying direct download...")
        os.system("wget -q http://cvit.iiit.ac.in/projects/IMFDB/IMFDB.zip -O IMFDB.zip 2>/dev/null")
        if os.path.exists("IMFDB.zip"):
            os.system("unzip -qo IMFDB.zip && rm IMFDB.zip")
            print("✅ IMFDB downloaded directly")
        else:
            print("⚠️ IMFDB download failed — training will use IndicFairFace only")
else:
    print("✅ IMFDB already exists")

imfdb_count = sum(1 for _ in Path("IMFDB").rglob("*.jpg")) if Path("IMFDB").exists() else 0
print(f"📊 IMFDB: {imfdb_count} images")

In [ ]:
# CELL 4: Face preprocessing
import cv2

def detect_align_crop(image_path, target_size=112):
    try:
        img = cv2.imread(str(image_path))
        if img is None: return None
        from retinaface import RetinaFace
        faces = RetinaFace.detect_faces(img)
        if not faces: return None
        face = max(faces.values(), key=lambda f: f['facial_area'][2]*f['facial_area'][3])
        lm = face['landmarks']
        le = np.array(lm['left_eye']); re = np.array(lm['right_eye'])
        ec = (le+re)/2
        angle = np.degrees(np.arctan2(re[1]-le[1], re[0]-le[0]))
        h,w = img.shape[:2]
        M = cv2.getRotationMatrix2D(tuple(ec.astype(int)), angle, 1.0)
        aligned = cv2.warpAffine(img, M, (w,h))
        x,y = ec.astype(int)-target_size//2
        x,y = max(0,min(x,w-target_size)), max(0,min(y,h-target_size))
        crop = aligned[y:y+target_size, x:x+target_size]
        if crop.shape[0]<target_size or crop.shape[1]<target_size: return None
        return crop.astype(np.float32)/255.0
    except: return None

print("✅ Face preprocessing ready")

In [ ]:
# CELL 5: Process all datasets
os.makedirs("data/faces", exist_ok=True)
total_processed = 0

# Process IndicFairFace
if indic_dir:
    for region in tqdm(sorted(Path(indic_dir).iterdir()), desc="IndicFairFace"):
        if not region.is_dir(): continue
        out = f"data/faces/{region.name}"
        os.makedirs(out, exist_ok=True)
        for img in region.rglob("*.jpg"):
            face = detect_align_crop(img)
            if face is not None:
                np.save(f"{out}/{total_processed:05d}.npy", face)
                total_processed += 1
    print(f"✅ IndicFairFace: {total_processed} faces")

# Process IMFDB
imfdb_start = total_processed
if Path("IMFDB").exists():
    for img in tqdm(list(Path("IMFDB").rglob("*.jpg"))[:30000], desc="IMFDB"):
        person = img.parent.name
        out = f"data/faces/{person}"
        os.makedirs(out, exist_ok=True)
        face = detect_align_crop(img)
        if face is not None:
            np.save(f"{out}/{total_processed:05d}.npy", face)
            total_processed += 1
    print(f"✅ IMFDB: {total_processed-imfdb_start} faces")

print(f"📊 Total: {total_processed} faces processed")
assert total_processed > 100, f"Only {total_processed} faces — check dataset downloads"

In [ ]:
# CELL 6: Data loaders
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.7),
    A.OneOf([A.RandomBrightnessContrast(0.15,0.15,p=1), A.CLAHE(4.0,p=1), A.ColorJitter(0.1,0.1,0.2,0.05,p=1)], p=0.5),
    A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2(),
])
val_tf = A.Compose([A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2()])

class FaceDS(Dataset):
    def __init__(self, root, tf=None):
        self.samples=[]; self.labels={}; li=0
        for p in sorted(Path(root).iterdir()):
            if not p.is_dir(): continue
            files=list(p.glob("*.npy"))
            if len(files)>=1:
                if p.name not in self.labels: self.labels[p.name]=li; li+=1
                for f in files: self.samples.append((f,self.labels[p.name]))
        self.num_classes=li; self.tf=tf
        print(f"  {len(self.samples)} samples, {li} classes")
    def __len__(self): return len(self.samples)
    def __getitem__(self,i):
        p,l=self.samples[i]; face=np.load(p)
        if self.tf: face=self.tf(image=face)['image']
        else: face=torch.from_numpy(face.transpose(2,0,1)).float()
        return face,l

print("📦 Loading datasets...")
train_ds=FaceDS("data/faces",train_tf); val_ds=FaceDS("data/faces",val_tf)
assert train_ds.num_classes>=2, f"Need >=2 classes, got {train_ds.num_classes}"
train_dl=DataLoader(train_ds,batch_size=64,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
val_dl=DataLoader(val_ds,batch_size=64,shuffle=False,num_workers=2,pin_memory=True)
print(f"✅ Ready: {len(train_ds)} train, {len(val_ds)} val")

In [ ]:
# CELL 7: MobileFaceNet + SE + CBAM
import torch.nn as nn
import torch.nn.functional as F

class SE(nn.Module):
    def __init__(self,ch,r=8):
        super().__init__(); self.pool=nn.AdaptiveAvgPool2d(1)
        self.fc1=nn.Conv2d(ch,ch//r,1,bias=False); self.fc2=nn.Conv2d(ch//r,ch,1,bias=False)
    def forward(self,x): return x*torch.sigmoid(self.fc2(F.relu(self.fc1(self.pool(x)))))

class CBAM(nn.Module):
    def __init__(self,ch,r=8):
        super().__init__()
        self.avg=nn.AdaptiveAvgPool2d(1); self.mx=nn.AdaptiveMaxPool2d(1)
        self.fc1=nn.Conv2d(ch,ch//r,1,bias=False); self.fc2=nn.Conv2d(ch//r,ch,1,bias=False)
        self.conv=nn.Conv2d(2,1,7,padding=3,bias=False)
    def forward(self,x):
        x=x*torch.sigmoid(self.fc2(F.relu(self.fc1(self.avg(x))))+self.fc2(F.relu(self.fc1(self.mx(x)))))
        a=torch.mean(x,1,keepdim=True); m,_=torch.max(x,1,keepdim=True)
        return x*torch.sigmoid(self.conv(torch.cat([a,m],1)))

class DWSep(nn.Module):
    def __init__(self,ic,oc,s=1):
        super().__init__()
        self.dw=nn.Conv2d(ic,ic,3,s,1,groups=ic,bias=False); self.bn1=nn.BatchNorm2d(ic)
        self.pw=nn.Conv2d(ic,oc,1,bias=False); self.bn2=nn.BatchNorm2d(oc)
    def forward(self,x): return F.relu(self.bn2(self.pw(F.relu(self.bn1(self.dw(x))))))

class MB(nn.Module):
    def __init__(self,ic,oc,s,er):
        super().__init__(); h=ic*er; self.res=(s==1 and ic==oc)
        ls=[]
        if er!=1: ls+=[nn.Conv2d(ic,h,1,bias=False),nn.BatchNorm2d(h),nn.PReLU(h)]
        ls+=[nn.Conv2d(h,h,3,s,1,groups=h,bias=False),nn.BatchNorm2d(h),nn.PReLU(h),SE(h),CBAM(h)]
        ls+=[nn.Conv2d(h,oc,1,bias=False),nn.BatchNorm2d(oc)]
        self.conv=nn.Sequential(*ls)
    def forward(self,x):
        o=self.conv(x); return o+x if self.res else o

class MobileFaceNet(nn.Module):
    def __init__(self,emb=128):
        super().__init__()
        self.c1=DWSep(3,64,2)
        self.s2=nn.Sequential(MB(64,64,1,2),MB(64,64,1,2))
        self.s3=nn.Sequential(MB(64,128,2,4),MB(128,128,1,4))
        self.s4=nn.Sequential(MB(128,128,2,4),MB(128,128,1,4))
        self.c2=DWSep(128,512,2)
        self.gdc=nn.Sequential(nn.Conv2d(512,512,7,groups=512,bias=False),nn.BatchNorm2d(512))
        self.lin=nn.Linear(512,emb,bias=False)
        self.bn=nn.BatchNorm1d(emb,affine=False)
        for m in self.modules():
            if isinstance(m,nn.Conv2d): nn.init.kaiming_normal_(m.weight,nonlinearity="relu")
            elif isinstance(m,(nn.BatchNorm2d,nn.BatchNorm1d)): m.weight.data.fill_(1); m.bias.data.zero_()
    def forward(self,x):
        x=self.c1(x); x=self.s2(x); x=self.s3(x); x=self.s4(x); x=self.c2(x)
        x=self.gdc(x); x=self.lin(x.flatten(1)); x=self.bn(x)
        n=torch.norm(x,2,1,True); return x/n, n

model=MobileFaceNet().cuda()
params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ MobileFaceNet: {params:,} params")
clear_gpu()

In [ ]:
# CELL 8: Train recognition (30 epochs)
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

class ArcFace(nn.Module):
    def __init__(self,ed,nc,m=0.5,s=64):
        super().__init__(); self.w=nn.Parameter(torch.Tensor(nc,ed)); nn.init.xavier_uniform_(self.w); self.m=m; self.s=s
    def forward(self,e,l):
        W=F.normalize(self.w,2,1); c=F.linear(e,W).clamp(-1+1e-7,1-1e-7)
        t=torch.zeros_like(c).scatter_(1,l.unsqueeze(1),1.0)
        return F.cross_entropy(torch.cos(torch.acos(c)+t*self.m)*self.s,l)

crit=ArcFace(128,train_ds.num_classes).cuda()
opt=AdamW(list(model.parameters())+list(crit.parameters()),lr=1e-3,weight_decay=1e-4)
sch=CosineAnnealingLR(opt,30,1e-6)
ckpt="checkpoints/best_recog.pth"
best=0
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt,weights_only=True)); print("✅ Resumed from checkpoint")

print("🚀 Training recognition...")
for ep in range(30):
    model.train(); tl=0; c=0; n=0
    for x,y in train_dl:
        x,y=x.cuda(),y.cuda(); e,_=model(x); loss=crit(e,y)
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        tl+=loss.item()
        with torch.no_grad():
            _,p=F.linear(e.detach(),F.normalize(crit.w,2,1)).max(1); c+=p.eq(y).sum().item(); n+=y.size(0)
        del e,loss
    sch.step(); ta=100.*c/n
    model.eval(); vc=0; vn=0
    with torch.no_grad():
        for x,y in val_dl:
            x,y=x.cuda(),y.cuda(); e,_=model(x)
            _,p=F.linear(e,F.normalize(crit.w,2,1)).max(1); vc+=p.eq(y).sum().item(); vn+=y.size(0)
            del e
    va=100.*vc/vn
    print(f"  Ep {ep+1:02d}/30 | Loss {tl/len(train_dl):.4f} | Train {ta:.1f}% | Val {va:.1f}%")
    if va>best: best=va; torch.save(model.state_dict(),ckpt)
    clear_gpu()
print(f"✅ Done! Best: {best:.1f}%")

In [ ]:
# CELL 9: Train liveness (20 epochs)
class LivNet(nn.Module):
    def __init__(self,nc=3):
        super().__init__()
        self.f=nn.Sequential(
            nn.Conv2d(3,32,3,2,1,bias=False),nn.BatchNorm2d(32),nn.PReLU(32),
            nn.Conv2d(32,64,3,2,1,groups=32,bias=False),nn.BatchNorm2d(64),nn.PReLU(64),
            nn.Conv2d(64,64,1,bias=False),nn.BatchNorm2d(64),nn.PReLU(64),
            nn.Conv2d(64,128,3,2,1,groups=64,bias=False),nn.BatchNorm2d(128),nn.PReLU(128),
            nn.Conv2d(128,128,1,bias=False),nn.BatchNorm2d(128),nn.PReLU(128),
            nn.Conv2d(128,256,3,2,1,groups=128,bias=False),nn.BatchNorm2d(256),nn.PReLU(256),
            nn.Conv2d(256,256,1,bias=False),nn.BatchNorm2d(256),nn.PReLU(256),
        )
        self.c=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(0.3),nn.Linear(256,64),nn.ReLU(True),nn.Dropout(0.15),nn.Linear(64,nc))
    def forward(self,x): return self.c(self.f(x))

liv=LivNet().cuda()
lcrit=nn.CrossEntropyLoss(weight=torch.tensor([1.,1.5,1.5]).cuda(),label_smoothing=0.1)
lopt=AdamW(liv.parameters(),lr=5e-4,weight_decay=1e-4)
lsch=CosineAnnealingLR(lopt,20,1e-6)

# Synthetic liveness data
class LivDS(Dataset):
    def __init__(self,n=5000): self.n=n
    def __len__(self): return self.n
    def __getitem__(self,i):
        l=i%3
        if l==0: img=np.random.rand(3,112,112).astype(np.float32)*0.6+0.2
        elif l==1: img=np.random.rand(3,112,112).astype(np.float32)*0.3+0.3
        else: img=np.random.rand(3,112,112).astype(np.float32)*0.8+0.1
        return torch.from_numpy(img),l

ldl=DataLoader(LivDS(),batch_size=64,shuffle=True,num_workers=2)
print("🚀 Training liveness...")
for ep in range(20):
    liv.train(); tl=0; c=0; n=0
    for x,y in ldl:
        x,y=x.cuda(),y.cuda(); o=liv(x); loss=lcrit(o,y)
        lopt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(liv.parameters(),5.0); lopt.step()
        tl+=loss.item(); _,p=o.max(1); c+=p.eq(y).sum().item(); n+=y.size(0); del o,loss
    lsch.step()
    print(f"  Ep {ep+1:02d}/20 | Loss {tl/len(ldl):.4f} | Acc {100.*c/n:.1f}%")
    clear_gpu()
torch.save(liv.state_dict(),"checkpoints/best_liv.pth")
print("✅ Liveness done!")

In [ ]:
# CELL 10: Export ONNX + TFLite
# Load best weights
if os.path.exists("checkpoints/best_recog.pth"): model.load_state_dict(torch.load("checkpoints/best_recog.pth",weights_only=True))
if os.path.exists("checkpoints/best_liv.pth"): liv.load_state_dict(torch.load("checkpoints/best_liv.pth",weights_only=True))

# ONNX export (CPU)
m_cpu=model.cpu().eval(); l_cpu=liv.cpu().eval(); dummy=torch.randn(1,3,112,112)
torch.onnx.export(m_cpu,dummy,"face_recognition.onnx",opset_version=13,input_names=["input"],output_names=["embedding"],dynamic_axes={"input":{0:"batch"},"embedding":{0:"batch"}})
torch.onnx.export(l_cpu,dummy,"liveness_detector.onnx",opset_version=13,input_names=["input"],output_names=["output"],dynamic_axes={"input":{0:"batch"},"output":{0:"batch"}})
print("✅ ONNX exported")

del dummy; model.cuda(); liv.cuda(); clear_gpu()

# TFLite
try:
    import tensorflow as tf
    for of,tf_f in [("face_recognition.onnx","face_recognition.tflite"),("liveness_detector.onnx","liveness_detector.tflite")]:
        sd=tf_f.replace('.tflite','_sm')
        try:
            import onnx2tf
            onnx2tf.convert(input_onnx_file_path=of,output_folder_path=sd,non_verbose=True)
        except: pass
        if os.path.exists(sd):
            try:
                c=tf.lite.TFLiteConverter.from_saved_model(sd)
                c.optimizations=[tf.lite.Optimize.DEFAULT]
                c.representative_dataset=lambda:([np.random.randn(1,3,112,112).astype(np.float32)] for _ in range(100))
                c.target_spec.supported_types=[tf.int8]
                m=c.convert()
                open(tf_f,"wb").write(m)
                print(f"  ✅ {tf_f} ({len(m)/1024/1024:.2f} MB)")
            except Exception as e: print(f"  ⚠️ {tf_f}: {e}")
except ImportError: print("⚠️ TensorFlow not available, skipping TFLite")

In [ ]:
# CELL 11: Summary + Save
print("="*50)
print("📊 NETRAEDGE TRAINING COMPLETE")
print(f"  Recognition: {params:,} params | Best: {best:.1f}%")
print(f"  Liveness: {sum(p.numel() for p in liv.parameters()):,} params")
for f in ["face_recognition.tflite","liveness_detector.tflite","face_recognition.onnx","liveness_detector.onnx"]:
    if os.path.exists(f): print(f"  {f}: {os.path.getsize(f)/1024/1024:.2f} MB")
print(f"  Faces: {total_processed}")
print("="*50)

# Save to Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/NetraEdge',exist_ok=True)
    for f in ["face_recognition.tflite","liveness_detector.tflite","face_recognition.onnx","liveness_detector.onnx"]:
        if os.path.exists(f): os.system(f"cp {f} /content/drive/MyDrive/NetraEdge/")
    print("✅ Saved to Google Drive/NetraEdge/")
except: pass

# Auto download
try:
    from google.colab import files
    import time; time.sleep(2)
    for f in ["face_recognition.tflite","liveness_detector.tflite"]:
        if os.path.exists(f): files.download(f); time.sleep(1)
    print("✅ Downloads triggered!")
except: pass